# 05 — Model Training and Evaluation

Trains:
1. Logistic Regression (baseline, interpretable).
2. LightGBM (production candidate), calibrated with isotonic regression.

Evaluates: AUC, log loss, top-decile churn capture, lift, calibration, SHAP.

Persists to DuckDB:
- `analytics.mart_user_risk_scores` — user-level probability + risk band.
- `analytics.mart_model_performance` — model comparison metrics.

In [ ]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, log_loss, brier_score_loss,
    precision_score, recall_score, f1_score, confusion_matrix,
)
import lightgbm as lgb

sns.set_theme(style='whitegrid')
DUCKDB_PATH = Path('../data/processed/kkbox.duckdb')
RANDOM_STATE = 42

## Load features

In [ ]:
con = duckdb.connect(str(DUCKDB_PATH))
df = con.execute('SELECT * FROM analytics.mart_user_churn_features').df()
print('Loaded', df.shape[0], 'users with', df.shape[1], 'columns')
df['is_churn'].value_counts(normalize=True)

## Define feature columns

Drop ID-like and target columns. Drop the latest_expire_date because all users have similar expiry dates (filtered by v2 selection).

In [ ]:
TARGET = 'is_churn'
DROP = ['msno', 'is_churn', 'registration_init_time', 'latest_transaction_date', 'latest_expire_date']

categorical = ['gender', 'age_band', 'registered_via', 'latest_payment_method_id']
numerical = [c for c in df.columns if c not in DROP + categorical]

X = df[numerical + categorical].copy()
y = df[TARGET].astype(int).values
for c in categorical:
    if c in X.columns:
        X[c] = X[c].astype('string').fillna('unknown').astype(str)
for c in numerical:
    if c in X.columns:
        X[c] = pd.to_numeric(X[c], errors='coerce')

# Remove columns that are fully null (frequent in short v2 windows)
all_null_numeric = [c for c in numerical if c in X.columns and X[c].isna().all()]
if all_null_numeric:
    X = X.drop(columns=all_null_numeric)
    numerical = [c for c in numerical if c not in all_null_numeric]
    print('Dropped fully-null numeric columns:', all_null_numeric)
print(f'Numerical features: {len(numerical)}')
print(f'Categorical features: {len(categorical)}')
print(f'Class balance: churn rate = {y.mean():.2%}')

## Train / validation / holdout split

70 / 15 / 15 stratified split on the v2 snapshot. The "true" time-based simulation is in notebook 07 (model monitoring).

In [ ]:
X_temp, X_holdout, y_temp, y_holdout = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=RANDOM_STATE)
print(f'Train:    {len(X_train):>7,}  churn rate {y_train.mean():.2%}')
print(f'Validate: {len(X_val):>7,}  churn rate {y_val.mean():.2%}')
print(f'Holdout:  {len(X_holdout):>7,}  churn rate {y_holdout.mean():.2%}')

## Preprocessing pipeline

In [ ]:
numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
])

categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', max_categories=20, sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe,     numerical),
    ('cat', categorical_pipe, categorical),
])

## Model 1 — Logistic Regression (baseline)

In [ ]:
lr_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf',  LogisticRegression(C=1.0, class_weight='balanced',
                                 max_iter=1000, random_state=RANDOM_STATE))
])

lr_pipe.fit(X_train, y_train)
lr_val_proba = lr_pipe.predict_proba(X_val)[:, 1]
lr_val_auc   = roc_auc_score(y_val, lr_val_proba)
lr_val_loss  = log_loss(y_val, lr_val_proba)
print(f'LR validation: AUC = {lr_val_auc:.4f}  log loss = {lr_val_loss:.4f}')

## Model 2 — LightGBM (production candidate)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# LightGBM handles categoricals natively with ordinal encoding
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_enc = X_train.copy()
X_val_enc   = X_val.copy()
X_holdout_enc = X_holdout.copy()
X_train_enc[categorical] = encoder.fit_transform(X_train[categorical].astype(str))
X_val_enc[categorical]   = encoder.transform(X_val[categorical].astype(str))
X_holdout_enc[categorical] = encoder.transform(X_holdout[categorical].astype(str))

scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

params = dict(
    objective='binary',
    metric='binary_logloss',
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=200,
    feature_fraction=0.9,
    bagging_fraction=0.9,
    bagging_freq=5,
    scale_pos_weight=scale_pos_weight,
    verbose=-1,
    random_state=RANDOM_STATE,
)

lgb_train = lgb.Dataset(X_train_enc, y_train, categorical_feature=categorical)
lgb_val   = lgb.Dataset(X_val_enc, y_val,   categorical_feature=categorical, reference=lgb_train)

lgb_model = lgb.train(
    params, lgb_train,
    num_boost_round=2000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

lgb_val_proba = lgb_model.predict(X_val_enc)
lgb_val_auc   = roc_auc_score(y_val, lgb_val_proba)
lgb_val_loss  = log_loss(y_val, lgb_val_proba)
print(f'\nLGBM validation (raw): AUC = {lgb_val_auc:.4f}  log loss = {lgb_val_loss:.4f}')

## Calibrate LightGBM probabilities (isotonic)

Calibrated probabilities can be used directly for revenue-at-risk computation.

In [ ]:
from sklearn.isotonic import IsotonicRegression
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(lgb_val_proba, y_val)
lgb_val_proba_cal = iso.predict(lgb_val_proba)
print(f'After calibration: log loss {log_loss(y_val, lgb_val_proba_cal):.4f}  '
      f'Brier {brier_score_loss(y_val, lgb_val_proba_cal):.4f}')

## Holdout evaluation

In [ ]:
lr_holdout_proba   = lr_pipe.predict_proba(X_holdout)[:, 1]
lgb_holdout_proba  = lgb_model.predict(X_holdout_enc)
lgb_holdout_proba_cal = iso.predict(lgb_holdout_proba)

def top_decile_metrics(y_true, scores):
    df_eval = pd.DataFrame({'y': y_true, 'p': scores}).sort_values('p', ascending=False)
    n = len(df_eval)
    top_n = max(int(0.1 * n), 1)
    captured = df_eval['y'].iloc[:top_n].sum()
    total_churn = df_eval['y'].sum()
    recall = captured / max(total_churn, 1)
    base_rate = total_churn / n
    lift = (captured / top_n) / max(base_rate, 1e-9)
    return recall, lift

rows = []
for name, p in [('LogReg', lr_holdout_proba),
                ('LGBM (raw)', lgb_holdout_proba),
                ('LGBM (calibrated)', lgb_holdout_proba_cal)]:
    auc = roc_auc_score(y_holdout, p)
    ll  = log_loss(y_holdout, p)
    bs  = brier_score_loss(y_holdout, p)
    rec10, lift10 = top_decile_metrics(y_holdout, p)
    rows.append({'model': name, 'auc': auc, 'log_loss': ll, 'brier': bs,
                 'recall_top_decile': rec10, 'lift_top_decile': lift10})
perf = pd.DataFrame(rows)
perf

## Calibration plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for name, p in [('LogReg', lr_holdout_proba),
                ('LGBM raw', lgb_holdout_proba),
                ('LGBM calibrated', lgb_holdout_proba_cal)]:
    frac_pos, mean_pred = calibration_curve(y_holdout, p, n_bins=10, strategy='quantile')
    ax.plot(mean_pred, frac_pos, marker='o', label=name)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction churned (observed)')
ax.set_title('Calibration curve (holdout)')
ax.legend()
plt.tight_layout()
plt.show()

## Lift chart

In [ ]:
def lift_curve(y_true, scores, deciles=10):
    df_e = pd.DataFrame({'y': y_true, 'p': scores}).sort_values('p', ascending=False).reset_index(drop=True)
    n = len(df_e)
    base_rate = df_e['y'].mean()
    pcts, lifts = [], []
    for k in range(1, deciles + 1):
        cut = int(n * k / deciles)
        rate = df_e['y'].iloc[:cut].mean()
        pcts.append(k * 10)
        lifts.append(rate / base_rate)
    return pcts, lifts

pcts, lifts = lift_curve(y_holdout, lgb_holdout_proba_cal)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(pcts, lifts, marker='o', color='#d62728')
ax.axhline(1.0, linestyle='--', color='gray')
ax.set_xlabel('Top X% of predicted users')
ax.set_ylabel('Lift over base rate')
ax.set_title('Lift chart — LightGBM calibrated (holdout)')
plt.tight_layout()
plt.show()

## SHAP feature importance

In [ ]:
import shap
sample = X_holdout_enc.sample(min(5000, len(X_holdout_enc)), random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(sample)
shap.summary_plot(shap_values, sample, plot_type='bar', max_display=15, show=True)

In [ ]:
shap.summary_plot(shap_values, sample, max_display=15, show=True)

## Score all users + assign risk bands

Score every labelled user (train + val + holdout) and assign risk bands per [docs/10](../docs/10_modeling_and_evaluation.md).

In [ ]:
X_all = df[numerical + categorical].copy()
X_all[categorical] = X_all[categorical].astype('object')
X_all_enc = X_all.copy()
X_all_enc[categorical] = encoder.transform(X_all[categorical].astype(str))

all_proba_raw = lgb_model.predict(X_all_enc)
all_proba     = iso.predict(all_proba_raw)

def assign_band(p):
    if p >= 0.80: return 'critical'
    if p >= 0.55: return 'high'
    if p >= 0.30: return 'medium'
    return 'low'

actions = {
    'critical': 'Personalised offer + CS contact',
    'high':     'Targeted renewal discount',
    'medium':   'Re-engagement push',
    'low':      'No active intervention',
}

scores_df = pd.DataFrame({
    'msno': df['msno'].values,
    'churn_probability': all_proba,
    'risk_band': [assign_band(p) for p in all_proba],
    'expected_revenue': df['latest_amount_paid'].fillna(0).values,
    'is_churn': df['is_churn'].values,
})
scores_df['revenue_at_risk'] = scores_df['churn_probability'] * scores_df['expected_revenue']
scores_df['recommended_action'] = scores_df['risk_band'].map(actions)

print(scores_df['risk_band'].value_counts())
scores_df.head()

## Persist to DuckDB

In [ ]:
con.execute('CREATE SCHEMA IF NOT EXISTS analytics')
con.register('scores_df', scores_df)
con.execute('CREATE OR REPLACE TABLE analytics.mart_user_risk_scores AS SELECT * FROM scores_df')

perf['model'] = perf['model'].astype(str)
con.register('perf_df', perf)
con.execute('CREATE OR REPLACE TABLE analytics.mart_model_performance AS SELECT * FROM perf_df')

print('Wrote analytics.mart_user_risk_scores:', len(scores_df))
print('Wrote analytics.mart_model_performance:', len(perf))

In [ ]:
con.close()
print('Done.')